In [0]:
# COMMAND ----------

from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

SILVER_TRANSACTIONS_PATH = (
    f"{S3_DELTA_PATH}/silver_transactions"
)


# ============================================================
# 2. READ BRONZE TRANSACTIONS
# ============================================================

bronze_transactions_df = spark.table(BRONZE_TABLE)

print(f"Bronze table : {BRONZE_TABLE}")
print(f"Record count : {bronze_transactions_df.count()}")

display(bronze_transactions_df.limit(10))

In [0]:
# COMMAND ----------

silver_transactions_df = (
    bronze_transactions_df

    .select(
        F.col("TX_ID")
            .cast("long")
            .alias("tx_id"),

        F.col("SENDER_ACCOUNT_ID")
            .cast("long")
            .alias("sender_account_id"),

        F.col("RECEIVER_ACCOUNT_ID")
            .cast("long")
            .alias("receiver_account_id"),

        F.upper(
            F.trim(F.col("TX_TYPE"))
        ).alias("tx_type"),

        F.col("TX_AMOUNT")
            .cast("double")
            .alias("tx_amount"),

        # Your dataset uses values like 0,1,2,3...
        # Treat this as an event/time-step for now.
        F.col("TIMESTAMP")
            .cast("long")
            .alias("event_time"),

        F.col("IS_FRAUD")
            .cast("boolean")
            .alias("is_fraud"),

        F.col("ALERT_ID")
            .cast("long")
            .alias("alert_id"),

        # Preserve ingestion-related metadata
        F.col("_rescued_data"),
        F.col("_ingested_timestamp"),
        F.col("_source_file")
    )
)

display(silver_transactions_df.limit(10))

In [0]:
# COMMAND ----------

silver_transactions_df = (
    silver_transactions_df

    # Mandatory transaction ID
    .filter(
        F.col("tx_id").isNotNull()
    )

    # Sender must exist
    .filter(
        F.col("sender_account_id").isNotNull()
    )

    # Receiver must exist
    .filter(
        F.col("receiver_account_id").isNotNull()
    )

    # Transaction amount must exist
    .filter(
        F.col("tx_amount").isNotNull()
    )

    # Transaction amount must be positive
    .filter(
        F.col("tx_amount") > 0
    )
)

In [0]:
# COMMAND ----------

silver_transactions_df = (
    silver_transactions_df
    .dropDuplicates(["tx_id"])
)

In [0]:
# COMMAND ----------

silver_transactions_df = (
    silver_transactions_df
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# COMMAND ----------

(
    silver_transactions_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        SILVER_TRANSACTIONS_PATH
    )
    .saveAsTable(SILVER_TABLE)
)

print("Silver Transactions table created successfully.")
print(f"Unity Catalog table : {SILVER_TABLE}")
print(f"S3 location         : {SILVER_TRANSACTIONS_PATH}")